In [16]:
import numpy as np
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
# source Gradioexp/bin/activate  for terminal activation

In [17]:

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


x_train shape: (60000, 28, 28, 1)
y_train shape: (60000,)
60000 train samples
10000 test samples


In [18]:
# Model parameters
num_classes = 10
input_shape = (28, 28, 1)

In [19]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

In [20]:
# model = keras.Sequential(
#     [
#         keras.layers.Input(shape=input_shape),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.MaxPooling2D(pool_size=(2, 2)),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.GlobalAveragePooling2D(),
#         keras.layers.Dropout(0.5),
#         keras.layers.Dense(num_classes, activation="softmax"),
#     ]
# )


In [21]:
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
    ],
)


In [23]:
batch_size = 128
epochs = 20

callbacks = [
    keras.callbacks.ModelCheckpoint(filepath="model_at_epoch_{epoch}.keras"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2),
]

model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.15,
    callbacks=callbacks,
)
score = model.evaluate(x_test, y_test, verbose=0)


Epoch 1/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - acc: 0.8885 - loss: 0.3743 - val_acc: 0.9597 - val_loss: 0.1367
Epoch 2/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - acc: 0.9543 - loss: 0.1562 - val_acc: 0.9697 - val_loss: 0.1011
Epoch 3/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - acc: 0.9659 - loss: 0.1128 - val_acc: 0.9768 - val_loss: 0.0815
Epoch 4/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.9731 - loss: 0.0877 - val_acc: 0.9780 - val_loss: 0.0747
Epoch 5/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - acc: 0.9760 - loss: 0.0747 - val_acc: 0.9777 - val_loss: 0.0712
Epoch 6/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - acc: 0.9818 - loss: 0.0586 - val_acc: 0.9784 - val_loss: 0.0750
Epoch 7/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - acc: 0.9829 - loss: 0.0533 - val_acc: 0.9796 - val_loss: 0.0725


In [24]:
model.save("final_model.keras")


In [25]:
model = keras.saving.load_model("final_model.keras")


In [26]:
predictions = model.predict(x_test)


313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 485us/step


In [27]:
y_pred = np.argmax(predictions, axis=1)

In [28]:
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy*100.:.4f}")

Accuracy: 97.9700


In [29]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred, digits=4))
cm = confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0     0.9759    0.9929    0.9843       980
           1     0.9843    0.9947    0.9895      1135
           2     0.9882    0.9719    0.9800      1032
           3     0.9690    0.9901    0.9794      1010
           4     0.9896    0.9684    0.9789       982
           5     0.9897    0.9697    0.9796       892
           6     0.9864    0.9823    0.9843       958
           7     0.9823    0.9728    0.9775      1028
           8     0.9656    0.9805    0.9730       974
           9     0.9674    0.9713    0.9693      1009

    accuracy                         0.9797     10000
   macro avg     0.9798    0.9795    0.9796     10000
weighted avg     0.9798    0.9797    0.9797     10000



In [30]:
print(cm)


[[ 973    1    1    1    0    1    1    1    1    0]
 [   0 1129    1    1    0    0    2    0    2    0]
 [   6    2 1003    3    2    0    1    6    9    0]
 [   0    0    0 1000    0    2    0    5    3    0]
 [   1    0    3    1  951    0    3    2    2   19]
 [   2    0    0   11    1  865    4    0    7    2]
 [   5    3    0    1    2    3  941    0    3    0]
 [   2    8    5    2    0    0    0 1000    2    9]
 [   4    0    2    3    2    1    2    2  955    3]
 [   4    4    0    9    3    2    0    2    5  980]]
